# Automatic Guitar Transcription with Deep Learning

Notebook-base em PyTorch inspirado no artigo de 2025.

Este notebook foi organizado em blocos: configuração, pré-processamento, dataset, modelo, treino e avaliação.

## 1. Imports e configurações

In [ ]:
import os
from pathlib import Path
import json
import numpy as np
import librosa
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import precision_recall_fscore_support
import mirdata
from pathlib import Path

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

CFG = {
    'sr': 22050,
    'hop_length': 512,
    'n_bins': 192,
    'bins_per_octave': 24,
    'segment_bars': 4,
    'batch_size': 4,
    'lr': 1e-4,
    'epochs': 10,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
}

NUM_STRINGS = 6
NUM_CLASSES = 21  # 0 mute, 1 open, 2..20 frets 1..19


## 2. Caminhos e metadados

A ideia aqui é você apontar para as pastas do GuitarSet ou do seu próprio dataset.


In [ ]:
DATA_ROOT = Path('data')
AUDIO_DIR = DATA_ROOT / 'audio'
MIDI_DIR = DATA_ROOT / 'midi'
PROC_DIR = DATA_ROOT / 'processed'
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Exemplo de lista de itens.
# Substitua por leitura real dos seus metadados.
items = []


## 3. Funções de áudio

Neste bloco usamos CQT como representação de entrada, seguindo o artigo.


In [ ]:
def load_audio(path, sr=22050):
    y, _ = librosa.load(path, sr=sr, mono=True)
    y = y / (np.max(np.abs(y)) + 1e-8)
    return y


def compute_cqt(y, sr=22050, hop_length=512, n_bins=192, bins_per_octave=24):
    cqt = librosa.cqt(
        y=y,
        sr=sr,
        hop_length=hop_length,
        n_bins=n_bins,
        bins_per_octave=bins_per_octave,
    )
    mag = np.abs(cqt)
    db = librosa.amplitude_to_db(mag, ref=np.max)
    return db.astype(np.float32)


## 4. Codificação das labels

A tablatura é tratada como uma saída estruturada por corda e por classe.


In [ ]:
def fret_to_class(fret):
    if fret is None:
        return 0
    if fret == 0:
        return 1
    return min(int(fret) + 1, 20)


def empty_tab(num_frames):
    return np.zeros((num_frames, NUM_STRINGS, NUM_CLASSES), dtype=np.float32)


def empty_note_tab(num_notes):
    return np.zeros((num_notes, NUM_STRINGS, NUM_CLASSES), dtype=np.float32)


## 5. Dataset PyTorch

Aqui você conecta áudio, CQT e labels. A parte de MIDI/tablatura real deve ser preenchida com o seu parser.


In [ ]:
class GuitarTranscriptionDataset(Dataset):
    def __init__(self, items):
        self.items = items

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        item = self.items[idx]
        y = load_audio(item['audio_path'], CFG['sr'])
        x = compute_cqt(y, CFG['sr'], CFG['hop_length'], CFG['n_bins'], CFG['bins_per_octave'])

        # Esperado: item['frame_tab'] e item['note_tab'] já prontos em numpy arrays.
        frame_y = item.get('frame_tab', np.zeros((x.shape[1], NUM_STRINGS, NUM_CLASSES), dtype=np.float32))
        note_y = item.get('note_tab', np.zeros((x.shape[1], NUM_STRINGS, NUM_CLASSES), dtype=np.float32))

        x = torch.tensor(x, dtype=torch.float32).unsqueeze(0)
        frame_y = torch.tensor(frame_y, dtype=torch.float32)
        note_y = torch.tensor(note_y, dtype=torch.float32)
        return x, frame_y, note_y


## 6. Modelo: CNN + bloco de atenção

Este é um esqueleto fiel ao espírito do artigo, mas mais simples para começar em PyTorch.


In [ ]:
class CNNEncoder(nn.Module):
    def __init__(self, in_channels=1, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
            nn.Conv2d(hidden, hidden * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden * 2),
            nn.ReLU(),
            nn.MaxPool2d((2, 2)),
        )

    def forward(self, x):
        return self.net(x)


class SimpleConformerBlock(nn.Module):
    def __init__(self, d_model=128, nhead=4):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.ReLU(),
            nn.Linear(d_model * 4, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)
        return x


class GuitarTranscriptionModel(nn.Module):
    def __init__(self, d_model=128):
        super().__init__()
        self.cnn = CNNEncoder()
        self.proj = nn.LazyLinear(d_model)
        self.block = SimpleConformerBlock(d_model=d_model, nhead=4)
        self.frame_head = nn.Linear(d_model, NUM_STRINGS * NUM_CLASSES)
        self.note_head = nn.Linear(d_model, NUM_STRINGS * NUM_CLASSES)

    def forward(self, x):
        z = self.cnn(x)
        b, c, f, t = z.shape
        z = z.permute(0, 3, 1, 2).contiguous().view(b, t, c * f)
        z = self.proj(z)
        z = self.block(z)
        frame_logits = self.frame_head(z)
        note_logits = self.note_head(z)
        return frame_logits, note_logits


## 7. Loss e métricas


In [ ]:
bce = nn.BCEWithLogitsLoss()

def compute_loss(frame_logits, note_logits, frame_y, note_y):
    frame_y = frame_y.view(frame_y.size(0), -1)
    note_y = note_y.view(note_y.size(0), -1)
    loss_frame = bce(frame_logits, frame_y)
    loss_note = bce(note_logits, note_y)
    return loss_frame + loss_note


def sigmoid_pred(logits, thr=0.5):
    return (torch.sigmoid(logits) > thr).float()


def compute_prf(y_true, y_pred):
    yt = y_true.detach().cpu().numpy().reshape(-1)
    yp = y_pred.detach().cpu().numpy().reshape(-1)
    p, r, f1, _ = precision_recall_fscore_support(yt, yp, average='binary', zero_division=0)
    return p, r, f1


## 8. DataLoader de exemplo

Substitua `items` pelos seus arquivos reais.


In [ ]:
# Cria o loader do GuitarSet
guitarset = mirdata.initialize("guitarset")

# Baixa o dataset para a pasta padrão do mirdata
guitarset.download()

In [ ]:
# Carrega os tracks
tracks = guitarset.load_tracks()

def build_items(tracks, audio_type="audio_mic", max_tracks=None):
    items = []
    count = 0

    for track_id, track in tracks.items():
        audio_path = getattr(track, f"{audio_type}_path", None)
        jams_path = getattr(track, "jams_path", None)

        if audio_path is None or jams_path is None:
            continue

        items.append({
            "track_id": track_id,
            "audio_path": audio_path,
            "jams_path": jams_path,
            "frame_tab": None,
            "note_tab": None,
        })

        count += 1
        if max_tracks is not None and count >= max_tracks:
            break

    return items


# Monte os itens usando o áudio do microfone
items = build_items(tracks, audio_type="audio_mic", max_tracks=10)

print(len(items))
print(items[0])

## 9. Loop de treino


In [19]:
train_ds = GuitarTranscriptionDataset(items)
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True)

model = GuitarTranscriptionModel().to(CFG['device'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'])


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for x, frame_y, note_y in loader:
        x = x.to(device)
        frame_y = frame_y.to(device)
        note_y = note_y.to(device)

        optimizer.zero_grad()
        frame_logits, note_logits = model(x)
        loss = compute_loss(frame_logits, note_logits, frame_y, note_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(len(loader), 1)


for epoch in range(CFG['epochs']):
    loss = train_one_epoch(model, train_loader, optimizer, CFG['device'])
    print(f'Epoch {epoch+1:02d} | loss={loss:.4f}')


RuntimeError: stack expects each tensor to be equal size, but got [1, 192, 1390] at entry 0 and [1, 192, 1263] at entry 1

## 10. Salvamento do modelo


In [ ]:
OUT_DIR = Path('output')
OUT_DIR.mkdir(exist_ok=True)
torch.save(model.state_dict(), OUT_DIR / 'guitar_transcription_model.pt')


## 11. Próximos passos

1. Implementar o parser real do MIDI por corda.
2. Criar os arrays `frame_tab` e `note_tab`.
3. Substituir o bloco simples por um Conformer mais completo.
4. Adicionar beat-informed quantisation.
5. Validar com GuitarSet e depois com um dataset externo.
